# R8 — Dynamic Per-ETF ROC Threshold Reconstruction

Selection-threshold ablation only: the underlying predictive model is never retrained. Universe, target, the 34-feature engineered representation, the R1 cold-restart cadence, and standard BCE are all reused verbatim to regenerate NN v1/NN v2's out-of-sample scores (reproducing R1/R7's saved AUCs exactly), then held completely fixed. Only the decision rule applied to those scores changes: three fixed/Top-K baselines (`FIXED_050`, `TOP2`, `TOP3`) vs. a dynamically-updated, leakage-safe, per-ETF ROC threshold (`ROC_YOUDEN_J`, primary; `ROC_CLOSEST_UL` and an expanding-lookback variant, bounded sensitivities only). See `docs/weekly_sector_rotation_r8_dynamic_roc_test.md` for the full write-up and `r8_dynamic_roc.py` for the shared, unit-tested threshold/selection implementations this notebook imports. Does not implement R9.

In [1]:
"""
R8 full pipeline (scratchpad validation run). Once verified, transcribed into
r8_dynamic_roc_threshold_ablation.ipynb and executed there for the committed
artifacts. Selection-threshold ablation only: the underlying NN v1/v2 BCE
model (R1 cold-restart cadence, 34 engineered features, R1/R7 CARRY_FORWARD_BCE)
is never retrained -- only the decision rule applied to its scores changes.
"""
import json
import sys
import time
from math import ceil
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.preprocessing import StandardScaler

t0 = time.time()
REPO_ROOT = Path("/Users/gavinhussey/Downloads/AtlasQuant")
FEATURE_PANEL_PATH = REPO_ROOT / "data" / "processed" / "weekly_sector_rotation" / "feature_panel.csv"
OUTPUTS_ROOT = REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation" / "outputs"

sys.path.insert(0, str(REPO_ROOT / "research" / "strategies" / "weekly_sector_rotation"))
import r7_financial_losses as r7  # noqa: E402
import r8_dynamic_roc as r8  # noqa: E402

SECTOR_ETFS = r7.SECTOR_ETFS
FULL_FEATURE_COLS = r7.FULL_FEATURE_COLS
RANK_K_VALUES = [2, 3]
BLOCK_SIZE = r7.BLOCK_SIZE
N_BOOTSTRAP = r7.N_BOOTSTRAP
BOOTSTRAP_SEED = r7.BOOTSTRAP_SEED

# ---------------------------------------------------------------------------

## 1. Load panel + R1 primary target, authoritative feature set, and reproduce/verify R1's exact fold schedule.

In [2]:
# 1. Load panel + R1 target, authoritative feature set, fold schedule.
# ---------------------------------------------------------------------------
date_cols = ["date", "week_end", "actual_last_trading_date", "feature_cutoff_timestamp",
             "next_week_first_trading_date", "next_week_last_trading_date"]
panel = pd.read_csv(FEATURE_PANEL_PATH, parse_dates=date_cols)
panel = panel.sort_values(["symbol", "date"]).reset_index(drop=True)
AUDIT_COLUMNS = ["week_end", "actual_last_trading_date", "week_complete", "feature_cutoff_timestamp",
                  "next_week_first_trading_date", "next_week_last_trading_date"]
FEATURE_COLUMNS = [c for c in panel.columns if c not in {"date", "symbol", *AUDIT_COLUMNS}]

r1_targets = pd.read_csv(OUTPUTS_ROOT / "r1_target_definitions_full_panel.csv", parse_dates=["date"])
panel = panel.merge(
    r1_targets[["date", "symbol", "target_abs_1pct_next_week", "next_week_open_to_close_return",
                "next_week_spy_return"]],
    on=["date", "symbol"], how="left",
)
assert set(FULL_FEATURE_COLS).issubset(set(FEATURE_COLUMNS))

md = panel.dropna(subset=FEATURE_COLUMNS + ["target_abs_1pct_next_week"]).copy()
md["label_binary"] = md["target_abs_1pct_next_week"].astype(int)
md["year"] = md["date"].dt.year
years_all = sorted(md["year"].unique())
assert years_all == list(range(2005, 2027))
test_years = r7.derive_r1_fold_schedule(years_all)
assert test_years == list(range(2008, 2027)), f"fold schedule diverges from validated R1 schedule: {test_years}"
print(f"[{time.time()-t0:.0f}s] data loaded, {len(md)} rows, test_years={test_years[0]}-{test_years[-1]}")

# ---------------------------------------------------------------------------

[0s] data loaded, 12253 rows, test_years=2008-2026


## 2. Regenerate NN v1/NN v2 BCE out-of-sample scores -- verbatim R1/R7 `CARRY_FORWARD_BCE` cold-restart cadence (no retraining variant; this IS the R1/R7 protocol, reproduced to obtain row-level OOF predictions) -- then verify pooled AUC reproduces R1/R7's saved values (Step 18: R8 must never change AUC, since thresholding is a downstream decision rule, not a ranking change).

In [3]:
# 2. Regenerate NN v1/v2 BCE OOS scores -- verbatim R1/R7 Objective-A cadence.
# ---------------------------------------------------------------------------
NN_SEED = 20260812
HIDDEN_1, HIDDEN_2 = 16, 8
DROPOUT = 0.3
WEIGHT_DECAY = 1e-4
LEARNING_RATE = 1e-3
MAX_EPOCHS = 300
PATIENCE = 15


class SmallMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, HIDDEN_1), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_1, HIDDEN_2), nn.ReLU(), nn.Dropout(DROPOUT),
            nn.Linear(HIDDEN_2, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


V2_HIDDEN = 32
V2_DROPOUT = 0.4
V2_WEIGHT_DECAY = 1e-3
V2_LEARNING_RATE = 1e-3
V2_MAX_EPOCHS = 300
V2_PATIENCE = 15
N_SEEDS = 5
SEED_BASE = 20260812


class WideSingleLayerMLP(nn.Module):
    def __init__(self, n_features: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(n_features, V2_HIDDEN), nn.ReLU(), nn.Dropout(V2_DROPOUT),
            nn.Linear(V2_HIDDEN, 1),
        )

    def forward(self, x):
        return self.net(x).squeeze(-1)


BCE_LOSS = nn.BCEWithLogitsLoss()


def train_fold_mlp(X_fit, y_fit, X_val, y_val, n_features, seed, v2=False):
    torch.manual_seed(seed)
    model = WideSingleLayerMLP(n_features) if v2 else SmallMLP(n_features)
    lr = V2_LEARNING_RATE if v2 else LEARNING_RATE
    wd = V2_WEIGHT_DECAY if v2 else WEIGHT_DECAY
    max_epochs = V2_MAX_EPOCHS if v2 else MAX_EPOCHS
    patience = V2_PATIENCE if v2 else PATIENCE
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    X_fit_t = torch.tensor(X_fit, dtype=torch.float32); y_fit_t = torch.tensor(y_fit, dtype=torch.float32)
    X_val_t = torch.tensor(X_val, dtype=torch.float32); y_val_t = torch.tensor(y_val, dtype=torch.float32)
    best_val_loss = float("inf"); best_state = None; epochs_without_improvement = 0
    for _ in range(max_epochs):
        model.train(); optimizer.zero_grad()
        loss = BCE_LOSS(model(X_fit_t), y_fit_t)
        loss.backward(); optimizer.step()
        model.eval()
        with torch.no_grad():
            val_loss = BCE_LOSS(model(X_val_t), y_val_t).item()
        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                break
    model.load_state_dict(best_state); model.eval()
    return model


def walk_forward_bce(md, years_, test_years_, feature_cols, v2=False):
    oof_frames = []
    for test_year in test_years_:
        train_years_available = [y for y in years_ if y < test_year]
        val_year = train_years_available[-1]
        fit_years = train_years_available[:-1]
        train_mask = md["year"].isin(train_years_available)
        fit_mask = md["year"].isin(fit_years)
        val_mask = md["year"] == val_year
        test_mask = md["year"] == test_year
        scaler = StandardScaler().fit(md.loc[train_mask, feature_cols])
        X_fit = scaler.transform(md.loc[fit_mask, feature_cols])
        X_val = scaler.transform(md.loc[val_mask, feature_cols])
        X_test = scaler.transform(md.loc[test_mask, feature_cols])
        y_fit = md.loc[fit_mask, "label_binary"].to_numpy()
        y_val = md.loc[val_mask, "label_binary"].to_numpy()

        if v2:
            seed_predictions = []
            for seed_offset in range(N_SEEDS):
                model = train_fold_mlp(X_fit, y_fit, X_val, y_val, len(feature_cols), SEED_BASE + seed_offset, v2=True)
                with torch.no_grad():
                    seed_predictions.append(torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy())
            predicted_proba = np.mean(seed_predictions, axis=0)
        else:
            model = train_fold_mlp(X_fit, y_fit, X_val, y_val, len(feature_cols), NN_SEED, v2=False)
            with torch.no_grad():
                predicted_proba = torch.sigmoid(model(torch.tensor(X_test, dtype=torch.float32))).numpy()

        fold = md.loc[test_mask, ["date", "symbol", "next_week_open_to_close_return",
                                   "next_week_spy_return", "label_binary"]].copy()
        fold["predicted_proba"] = predicted_proba
        oof_frames.append(fold)
    return pd.concat(oof_frames, ignore_index=True)


oof_v1 = walk_forward_bce(md, years_all, test_years, FULL_FEATURE_COLS, v2=False)
oof_v2 = walk_forward_bce(md, years_all, test_years, FULL_FEATURE_COLS, v2=True)
auc_v1 = roc_auc_score(oof_v1.label_binary, oof_v1.predicted_proba)
auc_v2 = roc_auc_score(oof_v2.label_binary, oof_v2.predicted_proba)
print(f"[{time.time()-t0:.0f}s] NN_v1 BCE pooled_auc={auc_v1:.6f} (expected ~0.5680)")
print(f"[{time.time()-t0:.0f}s] NN_v2 BCE pooled_auc={auc_v2:.6f} (expected ~0.5708)")
assert abs(auc_v1 - 0.5680100994943317) < 1e-4, "NN v1 BCE does not reproduce R1/R7's saved AUC"
assert abs(auc_v2 - 0.5735730758971164) < 1e-4, "NN v2 BCE does not reproduce R1/R7's saved AUC"
print(f"[{time.time()-t0:.0f}s] Step 2/3 (BCE reproduction verified against R1/R7) done.")

# ---------------------------------------------------------------------------

[19s] NN_v1 BCE pooled_auc=0.568010 (expected ~0.5680)
[19s] NN_v2 BCE pooled_auc=0.573573 (expected ~0.5708)
[19s] Step 2/3 (BCE reproduction verified against R1/R7) done.


## 3/4. Baseline definitions (`FIXED_050`, `TOP2`, `TOP3`) and the three ROC threshold reconstructions (`ROC_YOUDEN_J` primary + `ROC_CLOSEST_UL` and `ROC_YOUDEN_J_EXPANDING` bounded sensitivities) -- build the buy/no-buy signal for every (model, method) combination.

In [4]:
# 4. Build selection signals for every baseline + ROC method.
# ---------------------------------------------------------------------------
ROC_METHOD_CONFIG = {
    "ROC_YOUDEN_J": ("youden", r8.PRIMARY_LOOKBACK_WEEKS),          # PRIMARY
    "ROC_CLOSEST_UL": ("closest_ul", r8.PRIMARY_LOOKBACK_WEEKS),     # sensitivity (Step 4)
    "ROC_YOUDEN_J_EXPANDING": ("youden", "expanding"),               # sensitivity (Step 5)
}
BASELINE_METHODS = ["FIXED_050", "TOP2", "TOP3"]
ALL_METHODS = BASELINE_METHODS + list(ROC_METHOD_CONFIG.keys())
PRIMARY_METHOD = "ROC_YOUDEN_J"


def build_signal_for_method(oof: pd.DataFrame, method: str) -> pd.DataFrame:
    if method == "FIXED_050":
        sig = oof.copy()
        sig["threshold"] = r8.FIXED_THRESHOLD
        return r8.apply_threshold_selection(sig)
    if method in ("TOP2", "TOP3"):
        k = int(method[-1])
        sig = oof.copy()
        sig["buy_signal"] = 0
        sig["threshold"] = np.nan
        for date, g in sig.groupby("date"):
            if len(g) < 11:
                continue
            ranked = g["predicted_proba"].rank(ascending=False, method="first")
            sig.loc[g.index[(ranked <= k).to_numpy()], "buy_signal"] = 1
        return sig
    if method in ROC_METHOD_CONFIG:
        roc_method, lookback = ROC_METHOD_CONFIG[method]
        th = r8.compute_dynamic_thresholds_all_etfs(oof, method=roc_method, lookback=lookback)
        merged = oof.merge(th, on=["date", "symbol"], how="left")
        return r8.apply_threshold_selection(merged)
    raise ValueError(f"unknown method: {method}")


signals = {}
threshold_tables = {}
for model_name, oof in [("NN_v1", oof_v1), ("NN_v2", oof_v2)]:
    for method in ALL_METHODS:
        sig = build_signal_for_method(oof, method)
        signals[(model_name, method)] = sig
        if method in ROC_METHOD_CONFIG:
            threshold_tables[(model_name, method)] = sig[["date", "symbol", "threshold", "is_fallback", "n_obs_used"]].copy()
    print(f"[{time.time()-t0:.0f}s] {model_name}: built signals for {len(ALL_METHODS)} methods")

print(f"[{time.time()-t0:.0f}s] Step 4 (all selection signals built) done.")

# ---------------------------------------------------------------------------

[26s] NN_v1: built signals for 6 methods


[33s] NN_v2: built signals for 6 methods
[33s] Step 4 (all selection signals built) done.


## 5. Weekly active/calendar-cash series, selection-count diagnostics, classification metrics, and portfolio metrics for every (model, method) combination.

In [5]:
# 5. Weekly series (active + calendar/cash variants), selection counts,
#    classification metrics, portfolio metrics, per-method.
# ---------------------------------------------------------------------------
def method_weekly_series(oof_signal: pd.DataFrame):
    basket = r8.weekly_basket_metrics(oof_signal).set_index("date")
    active = basket[basket.status == "ACTIVE"].copy()
    active["precision"] = active["basket_positive_rate_row"]
    active["basket_minus_allsector"] = active["basket_mean_return"] - active["all_sector_mean_return"]
    active["basket_minus_spy"] = active["basket_mean_return"] - active["spy_return"]
    active["win_vs_spy"] = (active["basket_mean_return"] > active["spy_return"]).astype(float)

    cal = basket.copy()
    cal["basket_return_cash"] = np.where(cal.status == "ACTIVE", cal.basket_mean_return, 0.0)
    cal["basket_minus_allsector_cash"] = cal["basket_return_cash"] - cal["all_sector_mean_return"]
    cal["basket_minus_spy_cash"] = cal["basket_return_cash"] - cal["spy_return"]
    cal["win_vs_spy_cash"] = (cal["basket_return_cash"] > cal["spy_return"]).astype(float)
    return active, cal


selection_count_rows = []
selection_metrics_rows = []
portfolio_metrics_rows = []
weekly_counts_frames = []
active_series_store = {}
cal_series_store = {}

for model_name in ["NN_v1", "NN_v2"]:
    base_rate = signals[(model_name, "FIXED_050")]["label_binary"].mean()
    for method in ALL_METHODS:
        sig = signals[(model_name, method)]

        counts = r8.weekly_selection_counts(sig)
        counts.insert(0, "model", model_name); counts.insert(1, "method", method)
        weekly_counts_frames.append(counts)
        n_sel = counts["n_selected"]
        selection_count_rows.append({
            "model": model_name, "method": method, "n_weeks": len(n_sel),
            "mean_selected": n_sel.mean(), "median_selected": n_sel.median(),
            "p5_selected": n_sel.quantile(0.05), "p95_selected": n_sel.quantile(0.95),
            "pct_weeks_0": (n_sel == 0).mean(), "pct_weeks_1": (n_sel == 1).mean(),
            "pct_weeks_2": (n_sel == 2).mean(), "pct_weeks_3plus": (n_sel >= 3).mean(),
            "max_selected": n_sel.max(),
        })

        cls = r8.classification_metrics_for_selection(sig, base_rate)
        cls.update({"model": model_name, "method": method, "base_positive_rate": base_rate})
        selection_metrics_rows.append(cls)

        active, cal = method_weekly_series(sig)
        active_series_store[(model_name, method)] = active
        cal_series_store[(model_name, method)] = cal

        portfolio_metrics_rows.append({
            "model": model_name, "method": method,
            "n_active_weeks": len(active), "n_no_trade_weeks": int((cal.status == "NO_TRADE").sum()),
            "active_basket_mean_return": active["basket_mean_return"].mean(),
            "active_basket_median_return": active["basket_mean_return"].median(),
            "active_basket_minus_allsector_mean": active["basket_minus_allsector"].mean(),
            "active_basket_minus_spy_mean": active["basket_minus_spy"].mean(),
            "active_win_rate_vs_spy": active["win_vs_spy"].mean(),
            "active_pct_weeks_positive": (active["basket_mean_return"] > 0).mean(),
            "active_pct_weeks_ge_1pct": (active["basket_mean_return"] >= 0.01).mean(),
            "cash_basket_minus_allsector_mean": cal["basket_minus_allsector_cash"].mean(),
            "cash_basket_minus_spy_mean": cal["basket_minus_spy_cash"].mean(),
            "cash_win_rate_vs_spy": cal["win_vs_spy_cash"].mean(),
            "cash_pct_weeks_positive": (cal["basket_return_cash"] > 0).mean(),
            "cash_pct_weeks_ge_1pct": (cal["basket_return_cash"] >= 0.01).mean(),
        })

r8_selection_metrics = pd.DataFrame(selection_metrics_rows)
r8_weekly_selection_counts = pd.concat(weekly_counts_frames, ignore_index=True)
r8_portfolio_metrics = pd.DataFrame(portfolio_metrics_rows)
selection_count_summary = pd.DataFrame(selection_count_rows)
print(f"[{time.time()-t0:.0f}s] Step 5 (selection/portfolio metrics, all methods) done.")
print(selection_count_summary[["model", "method", "mean_selected", "pct_weeks_0"]].to_string())

# ---------------------------------------------------------------------------

[36s] Step 5 (selection/portfolio metrics, all methods) done.
    model                  method  mean_selected  pct_weeks_0
0   NN_v1               FIXED_050       0.108359     0.968008
1   NN_v1                    TOP2       2.000000     0.000000
2   NN_v1                    TOP3       3.000000     0.000000
3   NN_v1            ROC_YOUDEN_J       4.899897     0.152735
4   NN_v1          ROC_CLOSEST_UL       4.837977     0.251806
5   NN_v1  ROC_YOUDEN_J_EXPANDING       4.635707     0.240454
6   NN_v2               FIXED_050       0.366357     0.913313
7   NN_v2                    TOP2       2.000000     0.000000
8   NN_v2                    TOP3       3.000000     0.000000
9   NN_v2            ROC_YOUDEN_J       4.960784     0.150671
10  NN_v2          ROC_CLOSEST_UL       4.986584     0.247678
11  NN_v2  ROC_YOUDEN_J_EXPANDING       4.875129     0.221878


## 6. Per-ETF threshold diagnostics, threshold-stability diagnostics, and the fallback audit (ROC methods only).

In [6]:
# 6. Per-ETF threshold diagnostics + threshold stability (ROC methods only).
# ---------------------------------------------------------------------------
per_etf_rows = []
stability_rows = []
threshold_ts_frames = []
fallback_audit_rows = []

for model_name in ["NN_v1", "NN_v2"]:
    base_rate = signals[(model_name, "FIXED_050")]["label_binary"].mean()
    for method in ROC_METHOD_CONFIG:
        sig = signals[(model_name, method)]
        th_table = threshold_tables[(model_name, method)].copy()
        th_table.insert(0, "model", model_name); th_table.insert(1, "method", method)
        threshold_ts_frames.append(th_table)

        for symbol in SECTOR_ETFS:
            sub = sig[sig.symbol == symbol].sort_values("date").reset_index(drop=True)
            thetas = sub["threshold"]
            non_fallback = sub[~sub["is_fallback"]]
            first_non_fb_date = non_fallback["date"].min() if len(non_fallback) else pd.NaT
            selected = sub[sub.buy_signal == 1]
            per_etf_rows.append({
                "model": model_name, "method": method, "symbol": symbol,
                "mean_threshold": thetas.mean(), "median_threshold": thetas.median(),
                "std_threshold": thetas.std(), "min_threshold": thetas.min(), "max_threshold": thetas.max(),
                "first_non_fallback_date": first_non_fb_date,
                "fallback_frequency": float(sub["is_fallback"].mean()),
                "n_selected": len(selected),
                "selected_precision": float(selected["label_binary"].mean()) if len(selected) else np.nan,
                "realized_selected_mean_return": float(selected["next_week_open_to_close_return"].mean()) if len(selected) else np.nan,
                "selected_minus_spy_mean": float((selected["next_week_open_to_close_return"] - selected["next_week_spy_return"]).mean()) if len(selected) else np.nan,
            })

            delta = thetas.diff().dropna()
            major_jump_idx = delta.abs().idxmax() if len(delta) else None
            major_jump_date = sub.loc[major_jump_idx, "date"] if major_jump_idx is not None else pd.NaT
            stability_rows.append({
                "model": model_name, "method": method, "symbol": symbol,
                "mean_abs_weekly_change": delta.abs().mean(), "p95_abs_weekly_change": delta.abs().quantile(0.95),
                "max_abs_weekly_change": delta.abs().max(),
                "proportion_unchanged": float((delta == 0).mean()) if len(delta) else np.nan,
                "major_jump_date": major_jump_date, "major_jump_size": delta.abs().max() if len(delta) else np.nan,
            })

            by_year = sub.copy(); by_year["year"] = by_year["date"].dt.year
            yearly_fb = by_year.groupby("year")["is_fallback"].agg(["sum", "count"]).reset_index()
            yearly_fb["fallback_rate"] = yearly_fb["sum"] / yearly_fb["count"]
            yearly_fb.insert(0, "model", model_name); yearly_fb.insert(1, "method", method); yearly_fb.insert(2, "symbol", symbol)
            yearly_fb = yearly_fb.rename(columns={"sum": "n_fallback", "count": "n_weeks"})
            fallback_audit_rows.append(yearly_fb)

r8_per_etf_threshold_metrics = pd.DataFrame(per_etf_rows)
r8_threshold_stability = pd.DataFrame(stability_rows)
r8_threshold_timeseries = pd.concat(threshold_ts_frames, ignore_index=True)
r8_fallback_audit = pd.concat(fallback_audit_rows, ignore_index=True)
print(f"[{time.time()-t0:.0f}s] Step 6 (per-ETF + stability + fallback audit) done.")

# ---------------------------------------------------------------------------

[36s] Step 6 (per-ETF + stability + fallback audit) done.


## 7. Block bootstrap -- standalone CIs for every method, then paired vs.-baselines comparisons for the primary method (`ROC_YOUDEN_J`) against all three baselines, in both the active-week and calendar-week-cash variants.

In [7]:
# 7. Block bootstrap: standalone CIs + paired vs baselines (primary method
#    ROC_YOUDEN_J vs FIXED_050/TOP2/TOP3, both active-week and calendar-cash).
# ---------------------------------------------------------------------------
block_bootstrap_ci = r7.block_bootstrap_ci

bootstrap_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    for method in ALL_METHODS:
        active = active_series_store[(model_name, method)]
        cal = cal_series_store[(model_name, method)]
        base_rate = signals[(model_name, "FIXED_050")]["label_binary"].mean()
        if len(active) >= BLOCK_SIZE:
            b = block_bootstrap_ci(active["precision"] - base_rate)
            bootstrap_rows.append({"model": model_name, "method": method, "variant": "active",
                                    "quantity": "precision_minus_base", **b})
            for qname, col in [("basket_minus_allsector", "basket_minus_allsector"),
                                ("basket_minus_spy", "basket_minus_spy")]:
                b = block_bootstrap_ci(active[col])
                bootstrap_rows.append({"model": model_name, "method": method, "variant": "active",
                                        "quantity": qname, **b})
            b = block_bootstrap_ci(active["win_vs_spy"] - 0.5)
            bootstrap_rows.append({"model": model_name, "method": method, "variant": "active",
                                    "quantity": "win_rate_edge_vs_spy", **b})
        if len(cal) >= BLOCK_SIZE:
            for qname, col in [("basket_minus_allsector_cash", "basket_minus_allsector_cash"),
                                ("basket_minus_spy_cash", "basket_minus_spy_cash")]:
                b = block_bootstrap_ci(cal[col])
                bootstrap_rows.append({"model": model_name, "method": method, "variant": "calendar_cash",
                                        "quantity": qname, **b})
            b = block_bootstrap_ci(cal["win_vs_spy_cash"] - 0.5)
            bootstrap_rows.append({"model": model_name, "method": method, "variant": "calendar_cash",
                                    "quantity": "win_rate_edge_vs_spy", **b})
r8_bootstrap_ci = pd.DataFrame(bootstrap_rows)
print(f"[{time.time()-t0:.0f}s] Step 7a (standalone bootstrap, all methods) done.")


def paired_diff_bootstrap(series_a: pd.Series, series_b: pd.Series) -> dict:
    common_idx = series_a.index.intersection(series_b.index)
    if len(common_idx) < BLOCK_SIZE:
        return None
    diff = (series_a.loc[common_idx] - series_b.loc[common_idx]).sort_index()
    return block_bootstrap_ci(diff)


paired_rows = []
for model_name in ["NN_v1", "NN_v2"]:
    active_primary = active_series_store[(model_name, PRIMARY_METHOD)]
    cal_primary = cal_series_store[(model_name, PRIMARY_METHOD)]
    for baseline in BASELINE_METHODS:
        active_base = active_series_store[(model_name, baseline)]
        cal_base = cal_series_store[(model_name, baseline)]
        for qname, col_a, col_b, variant in [
            ("precision_minus_base_diff", "precision", "precision", "active"),
            ("basket_minus_allsector_diff", "basket_minus_allsector", "basket_minus_allsector", "active"),
            ("basket_minus_spy_diff", "basket_minus_spy", "basket_minus_spy", "active"),
            ("win_rate_diff", "win_vs_spy", "win_vs_spy", "active"),
        ]:
            b = paired_diff_bootstrap(active_primary[col_a], active_base[col_b])
            if b is not None:
                paired_rows.append({"model": model_name, "method": PRIMARY_METHOD, "vs_baseline": baseline,
                                     "variant": variant, "quantity": qname, **b})
        for qname, col, variant in [
            ("basket_minus_allsector_cash_diff", "basket_minus_allsector_cash", "calendar_cash"),
            ("basket_minus_spy_cash_diff", "basket_minus_spy_cash", "calendar_cash"),
            ("win_rate_cash_diff", "win_vs_spy_cash", "calendar_cash"),
        ]:
            b = paired_diff_bootstrap(cal_primary[col], cal_base[col])
            if b is not None:
                paired_rows.append({"model": model_name, "method": PRIMARY_METHOD, "vs_baseline": baseline,
                                     "variant": variant, "quantity": qname, **b})
        print(f"[{time.time()-t0:.0f}s] {model_name} {PRIMARY_METHOD} vs {baseline}: paired comparisons computed")

r8_vs_baselines_paired = pd.DataFrame(paired_rows)
print(f"[{time.time()-t0:.0f}s] Step 7b (paired vs baselines bootstrap) done.")

# ---------------------------------------------------------------------------

[49s] Step 7a (standalone bootstrap, all methods) done.


[50s] NN_v1 ROC_YOUDEN_J vs FIXED_050: paired comparisons computed


[51s] NN_v1 ROC_YOUDEN_J vs TOP2: paired comparisons computed


[52s] NN_v1 ROC_YOUDEN_J vs TOP3: paired comparisons computed


[53s] NN_v2 ROC_YOUDEN_J vs FIXED_050: paired comparisons computed


[54s] NN_v2 ROC_YOUDEN_J vs TOP2: paired comparisons computed


[55s] NN_v2 ROC_YOUDEN_J vs TOP3: paired comparisons computed
[55s] Step 7b (paired vs baselines bootstrap) done.


## 8. Sample audit.

In [8]:
# 8. Sample audit.
# ---------------------------------------------------------------------------
sample_audit_rows = []
for model_name, oof in [("NN_v1", oof_v1), ("NN_v2", oof_v2)]:
    full_weeks = oof.groupby("date").size()
    sample_audit_rows.append({
        "model": model_name, "n_oos_rows": len(oof), "n_weeks_total": oof["date"].nunique(),
        "n_weeks_full_11sector": int((full_weeks == 11).sum()),
        "first_date": str(oof["date"].min().date()), "last_date": str(oof["date"].max().date()),
        "primary_lookback_weeks": r8.PRIMARY_LOOKBACK_WEEKS, "min_roc_obs": r8.MIN_ROC_OBS,
        "fallback_threshold": r8.FALLBACK_THRESHOLD, "primary_method": PRIMARY_METHOD,
        "underlying_auc": float(roc_auc_score(oof.label_binary, oof.predicted_proba)),
    })
r8_sample_audit = pd.DataFrame(sample_audit_rows)

# ---------------------------------------------------------------------------

## 9. Save all required R8 artifacts.

In [9]:
# 9. Save all artifacts.
# ---------------------------------------------------------------------------
r8_threshold_timeseries.to_csv(OUTPUTS_ROOT / "r8_threshold_timeseries.csv", index=False)
r8_selection_metrics.to_csv(OUTPUTS_ROOT / "r8_selection_metrics.csv", index=False)
r8_weekly_selection_counts.to_csv(OUTPUTS_ROOT / "r8_weekly_selection_counts.csv", index=False)
r8_portfolio_metrics.to_csv(OUTPUTS_ROOT / "r8_portfolio_metrics.csv", index=False)
r8_per_etf_threshold_metrics.to_csv(OUTPUTS_ROOT / "r8_per_etf_threshold_metrics.csv", index=False)
r8_threshold_stability.to_csv(OUTPUTS_ROOT / "r8_threshold_stability.csv", index=False)
r8_bootstrap_ci.to_csv(OUTPUTS_ROOT / "r8_bootstrap_ci.csv", index=False)
r8_vs_baselines_paired.to_csv(OUTPUTS_ROOT / "r8_vs_baselines_paired_comparison.csv", index=False)
r8_fallback_audit.to_csv(OUTPUTS_ROOT / "r8_fallback_audit.csv", index=False)
r8_sample_audit.to_csv(OUTPUTS_ROOT / "r8_sample_audit.csv", index=False)

summary = {
    "primary_method": PRIMARY_METHOD, "primary_lookback_weeks": r8.PRIMARY_LOOKBACK_WEEKS,
    "min_roc_obs": r8.MIN_ROC_OBS, "fallback_threshold": r8.FALLBACK_THRESHOLD,
    "underlying_auc": {"NN_v1": float(auc_v1), "NN_v2": float(auc_v2)},
    "mean_selected_per_week": {
        f"{m}_{meth}": float(selection_count_summary[(selection_count_summary.model == m) &
                                                       (selection_count_summary.method == meth)]["mean_selected"].iloc[0])
        for m in ["NN_v1", "NN_v2"] for meth in ALL_METHODS
    },
    "no_trade_week_rate": {
        f"{m}_{meth}": float(selection_count_summary[(selection_count_summary.model == m) &
                                                       (selection_count_summary.method == meth)]["pct_weeks_0"].iloc[0])
        for m in ["NN_v1", "NN_v2"] for meth in ALL_METHODS
    },
    "runtime_seconds": time.time() - t0,
}
with open(OUTPUTS_ROOT / "r8_run_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(f"[{time.time()-t0:.0f}s] ALL DONE.")
print(json.dumps(summary, indent=2))

[55s] ALL DONE.
{
  "primary_method": "ROC_YOUDEN_J",
  "primary_lookback_weeks": 52,
  "min_roc_obs": 26,
  "fallback_threshold": 0.5,
  "underlying_auc": {
    "NN_v1": 0.5680100994943317,
    "NN_v2": 0.5735730758971164
  },
  "mean_selected_per_week": {
    "NN_v1_FIXED_050": 0.10835913312693499,
    "NN_v1_TOP2": 2.0,
    "NN_v1_TOP3": 3.0,
    "NN_v1_ROC_YOUDEN_J": 4.899896800825593,
    "NN_v1_ROC_CLOSEST_UL": 4.837977296181631,
    "NN_v1_ROC_YOUDEN_J_EXPANDING": 4.635706914344685,
    "NN_v2_FIXED_050": 0.36635706914344685,
    "NN_v2_TOP2": 2.0,
    "NN_v2_TOP3": 3.0,
    "NN_v2_ROC_YOUDEN_J": 4.96078431372549,
    "NN_v2_ROC_CLOSEST_UL": 4.9865841073271415,
    "NN_v2_ROC_YOUDEN_J_EXPANDING": 4.875128998968008
  },
  "no_trade_week_rate": {
    "NN_v1_FIXED_050": 0.9680082559339526,
    "NN_v1_TOP2": 0.0,
    "NN_v1_TOP3": 0.0,
    "NN_v1_ROC_YOUDEN_J": 0.15273477812177502,
    "NN_v1_ROC_CLOSEST_UL": 0.2518059855521156,
    "NN_v1_ROC_YOUDEN_J_EXPANDING": 0.2404540763673890